In [18]:
!pip install -q pythainlp deepcut

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.8 MB/s eta 0:00:00a 0:00:01


In [1]:
!pip install -q pythainlp nlpo3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 61.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.1 MB/s eta 0:00:0000:010:00:01


In [ ]:
!pip install -q pythainlp sefr_cut

In [ ]:
!pip install -q pythainlp attacut

In [2]:
!gdown 1uCjU13JqXxuUsErE3SPTB7VxL1pfNqec

Downloading...
From (original): https://drive.google.com/uc?id=1uCjU13JqXxuUsErE3SPTB7VxL1pfNqec
From (redirected): https://drive.google.com/uc?id=1uCjU13JqXxuUsErE3SPTB7VxL1pfNqec&confirm=t&uuid=1078430f-ee31-41cc-8a05-23d411352e04
To: /kaggle/working/AIFORTHAI LST20 Corpus.tar.gz
100%|██████████████████████████████████████| 13.6M/13.6M [00:00<00:00, 26.1MB/s]


In [3]:
!tar -xzf '/kaggle/working/AIFORTHAI LST20 Corpus.tar.gz'

In [19]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from tqdm.auto import tqdm

from pythainlp.tokenize import word_tokenize

from datasets import load_dataset
from marisa_trie import Trie
import deepcut

2026-04-04 09:52:19.565390: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775296339.861992      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775296339.947802      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
lst20_dataset = load_dataset('lst-nectec/lst20', trust_remote_code = True, data_dir = "/kaggle/working/LST20_Corpus")

README.md: 0.00B [00:00, ?B/s]

lst20.py: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/63310 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5620 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5250 [00:00<?, ? examples/s]

In [7]:
word_set = set()
splits = ['train', 'validation', 'test']
for split in splits:
    for words in lst20_dataset[split]['tokens']:
        for word in words:
            word_set.add(word)

print(f"All word: {len(word_set)}")
lst20_trie = Trie(word_set)

All word: 54770


In [8]:
with open("/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_test.txt","r") as f:
    test_doc = f.read()

with open("/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_list.txt") as f:
    ws_list = eval(f.read())

In [9]:
sample_sub = pd.read_csv("/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_sample_submission.csv")
sample_sub

,Id,Predicted
0,1,B_WORD
1,2,I_WORD
2,3,E_WORD
3,4,NaN
4,5,NaN
...,...,...
35177,37244,NaN
35178,37245,NaN
35179,37246,NaN
35180,37247,NaN


In [10]:
len(test_doc.replace(' ', ''))

35182

In [11]:
test_doc_clean = test_doc.replace(' ', '')

In [12]:
more_vocab = [
            "ทรลักษ์", "อนุพงษ์เผ่าจินดา", "ลองวิสาโล", "เนียงพาด", "พอลซาเรือน", "จีบีซี", "วลิตโรจน", "วลิต", "วิตก", "วีรชัย", "พลาศรัย", "เอี่ยมฐานนท์",
              "โมหำหมัด", "ซอรีสะมะแอ", "สุวรรณเกษการ", "ซอรี", "กวินตรา", "โพธิจักร", "ภู่ขวัญเมือง", "ไฟติ้ง", "เจริญวัชรวิทย์", "ปลั่งพินิจ", "ลุกทุ่ง", "มหาชน",
              "เมโท", "หยิง", "ตาราบารอฟ", "เฮย", "อุสซูริส", "อามูร์มา", "เหยหลง", "วูซูลิเจียง", "วานิจ", "ปิณฑวนิช", "โกสัย", 'ดาตอร์ปิโด', 'ดารณี', 'ศิลปกุล',
              'ยกุล', 'สืบวงษ์ลี', 'กระจอก', 'เจียโก', 'วิรายุดา', 'วงษ์เทศ', 'ขอม', 'ธม', 'เสภา', 'เสภา', 'ออเคสตรา' ,'อรรถจักร', 'สัตยานุรักษ์'
             ]
lst20_words = list(word_set)
lst20_words += more_vocab
lst20_trie = Trie(set(lst20_words))

In [14]:
from pythainlp.util import Trie

# This is the correct wrapper that PyThaiNLP engines expect
lst20_trie = Trie(lst20_words)

# Now your word_tokenize call will work without the prefixes() error
# new_words = word_tokenize(word, engine='newmm', custom_dict=lst20_trie)

In [20]:
import time
from pythainlp.tokenize import word_tokenize
from pythainlp.util import Trie

# Initialize engines (optional pre-load for speed)
# word_tokenize("warmup", engine='wangchanglm') 

id_count = 0
# 👇 Primary engine: Transformer-based (best context handling)
words = word_tokenize(test_doc, engine='nlpo3', keep_whitespace=False)
all_space_idx = [idx for idx, c in enumerate(test_doc) if c == " "]
encode_pairs = []
oov_words = []

words_clear = []
for word in words:
    words_clear += word.split(" ")

def has_numbers(s): return any(c.isdigit() for c in s)
def has_english(s): return any(c.isascii() and c.isalpha() for c in s)

for word in tqdm(words_clear, total=len(words_clear)):
    if word not in lst20_trie:
        oov_words.append(word)
        # 🔹 Smart fallback chain for OOV
        if has_numbers(word) or has_english(word):
            new_words = word_tokenize(word, engine='longest', custom_dict=lst20_trie)
        else:
            # Try sefr_cut (fast, handles compounds well), fall back to newmm
            new_words = word_tokenize(word, engine='deepcut')
            if len(new_words) == 1:  # Still failed
                new_words = word_tokenize(word, engine='newmm', custom_dict=lst20_trie)
    else:
        new_words = [word]

    # 🔹 Encode BIO tags (identical to your original logic)
    for n_word in new_words:
        for idx, c in enumerate(n_word):
            id_count += 1
            if id_count in all_space_idx:
                id_count += 1
            
            if idx == 0:
                encode_pairs.append([id_count, 'B_WORD'])
            elif idx == len(n_word) - 1 and len(n_word) > 1:
                encode_pairs.append([id_count, 'E_WORD'])
            else:
                encode_pairs.append([id_count, 'I_WORD'])

  0%|          | 0/7453 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-04-04 09:52:34.113475: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━

In [21]:
pred = pd.DataFrame(encode_pairs)
pred.columns = ['Id', 'Predicted']

In [22]:
sample_sub['Predicted'] = pred['Predicted']

In [23]:
sample_sub

,Id,Predicted
0,1,B_WORD
1,2,I_WORD
2,3,E_WORD
3,4,B_WORD
4,5,I_WORD
...,...,...
35177,37244,E_WORD
35178,37245,B_WORD
35179,37246,I_WORD
35180,37247,I_WORD


In [24]:
sample_sub.to_csv("nlpo3-deepcut-newmm-longest-moreVocab-space-custom_dict.csv", index=False)